# Import FRED data
- Author: Bryan Bravo
- Created: 2026-03-02
## Import Libraries

In [21]:
import os 
import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

### Custom Functions

In [17]:
def fetch_fred_data(
    api_key: str,
    currency_name: str, # Currency name of series id.
    series_id: str,  # FRED series ID
    start_date: str = "1980-01-01",
    end_date: str = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
    ) -> pd.DataFrame:
    """
    Fetches daily foreign exchange rate data from the FRED API, cleans it, and returns
    a standardized DataFrame containing the latest revision for each date.

    This function:
    - Adds a `currency` column for identification
    - Computes a unified USD exchange rate (`us_fx_rate`) so that:
        • If the series is FX-per-USD (e.g., DEXJPUS), it inverts the value
        • Otherwise, it uses the value as-is

    Parameters
    ----------
    api_key : str
        FRED API key used for authentication.
    currency_name : str
        Human-readable currency name to attach to the output (e.g., "euro", "yen").
    series_id : str
        FRED H.10 series ID (e.g., "DEXUSEU", "DEXJPUS").
    start_date : str
        Start date for the query in YYYY-MM-DD format. (Defaults to "1980-01-01")
    end_date : str
        End date for the query in YYYY-MM-DD format. (Defaults to the last day of previous month)

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - date : datetime64
        - currency : str
        - us_fx_rate : float
        representing the latest available FX rate for each date.

    Raises
    ------
    Exception
        If the API request fails or the response cannot be parsed.

    """

    try:
        response = requests.get(
            f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&realtime_start={start_date}&realtime_end={end_date}&api_key={api_key}&file_type=json",
            timeout=10)
        fred_data = response.json()

        if 'error_code' in fred_data:  # check if response is None
            print(f"✗ FRED API error: {fred_data.get('error_code', 'error_message')}")

        # Create pandas DataFrame from observations
        df = pd.DataFrame(fred_data['observations'])

        # Select relevant columns
        df = df[['date', 'realtime_start', 'value']]
        df['currency'] = currency_name

        # Correct data types
        df['date'] = pd.to_datetime(df['date'])
        df['realtime_start'] = pd.to_datetime(df['realtime_start'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')  # Convert to numeric, coerce errors to NaN

        # Add USD exchange rate (USD/FX)
        if series_id[5:] == "US":
            df['fx_rate'] = 1 / df['value']
        else:
            df['fx_rate'] = df['value']

        # Keep only the most recent revision for each date and subset columns
        df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
        df = df[['date', 'currency', 'fx_rate']].reset_index(drop=True)

        print(f"✓ Fetched {len(df)} currency records for '{currency_name}' from FRED for years {dt.strftime(df['date'].min(), '%Y-%m-%d')} through {dt.strftime(df['date'].max(), '%Y-%m-%d')}")
        return df
    except Exception as e:
        print(f"✗ Error fetching FRED data on {currency_name}: {str(e)[:100]}")

## Query

In [27]:
fred_fx_series = {
    "euro": "DEXUSEU",
    "japan": "DEXJPUS",
    "china": "DEXCHUS",
    "canada": "DEXCAUS",
    "united_kingdom": "DEXUSUK",
    "south_korea": "DEXKOUS",
    "mexico": "DEXMXUS",
    "india": "DEXINUS",
    "brazil": "DEXBZUS",
    "australia": "DEXUSAL",
    "switzerland": "DEXSZUS",
    "sweden": "DEXSDUS",
    "norway": "DEXNOUS",
    "denmark": "DEXDNUS",
    "hong_kong": "DEXHKUS",
    "singapore": "DEXSIUS",
    "south_africa": "DEXSFUS",
    "taiwan": "DEXTAUS",
    "thailand": "DEXTHUS",
    "new_zealand": "DEXUSNZ",
    "malaysia": "DEXMAUS",
    "venezuela": "DEXVZUS",
    "united_states": "DTWEXBGS"  # Nominal Broud U.S. Dollar Index
}

fred_data_df = reduce(lambda x, y: pd.concat([x, y], ignore_index=True),
                      [fetch_fred_data(hardcoded_keys.FRED_API_KEY, cur, ser) for cur, ser in fred_fx_series.items()])
fred_data_df.sort_values(['currency', 'date'], ascending=[True, True])


✓ Fetched 7080 currency records for 'euro' from FRED for years 1999-01-04 through 2026-02-20
✓ Fetched 14385 currency records for 'japan' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 11776 currency records for 'china' from FRED for years 1981-01-02 through 2026-02-20
✓ Fetched 14385 currency records for 'canada' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 14385 currency records for 'united_kingdom' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 11705 currency records for 'south_korea' from FRED for years 1981-04-13 through 2026-02-20
✓ Fetched 8425 currency records for 'mexico' from FRED for years 1993-11-08 through 2026-02-20
✓ Fetched 13864 currency records for 'india' from FRED for years 1973-01-02 through 2026-02-20
✓ Fetched 8125 currency records for 'brazil' from FRED for years 1995-01-02 through 2026-02-20
✓ Fetched 14385 currency records for 'australia' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 14385 currency records

,date,currency,fx_rate
104130,1971-01-04,australia,1.112700
104131,1971-01-05,australia,1.113200
104132,1971-01-06,australia,1.114000
104133,1971-01-07,australia,1.113800
104134,1971-01-08,australia,1.112400
...,...,...,...
273718,2026-02-16,venezuela,NaN
273719,2026-02-17,venezuela,0.002546
273720,2026-02-18,venezuela,0.002546
273721,2026-02-19,venezuela,0.002514


In [29]:
fred_data_df[fred_data_df['currency']=="united_states"]

,date,currency,fx_rate,fx_return,vol_30d,vol_30d_annualized
273723,2006-01-02,united_states,101.4155,10.614006,NaN,NaN
273724,2006-01-03,united_states,100.7558,-0.006526,NaN,NaN
273725,2006-01-04,united_states,100.2288,-0.005244,NaN,NaN
273726,2006-01-05,united_states,100.2992,0.000702,NaN,NaN
273727,2006-01-06,united_states,100.0241,-0.002747,NaN,NaN
...,...,...,...,...,...,...
278973,2026-02-16,united_states,NaN,NaN,NaN,NaN
278974,2026-02-17,united_states,117.7375,NaN,NaN,NaN
278975,2026-02-18,united_states,117.8426,0.000892,NaN,NaN
278976,2026-02-19,united_states,118.2354,0.003328,NaN,NaN


In [26]:
fred_data_df.info()
fred_data_df.value_counts('currency').sort_index()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278978 entries, 0 to 278977
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   date                278978 non-null  datetime64[ns]
 1   currency            278978 non-null  object        
 2   fx_rate             264092 non-null  float64       
 3   fx_return           253265 non-null  float64       
 4   vol_30d             59360 non-null   float64       
 5   vol_30d_annualized  59360 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 12.8+ MB


currency
australia         14385
brazil             8125
canada            14385
china             11776
denmark           14385
euro               7080
hong_kong         11776
india             13864
japan             14385
malaysia          14385
mexico             8425
new_zealand       14385
norway            14385
singapore         11776
south_africa      14385
south_korea       11705
sweden            14385
switzerland       14385
taiwan            11060
thailand          11776
united_kingdom    14385
united_states      5255
venezuela          8125
Name: count, dtype: int64